# Income Statement Analysis (CFA Level 1)

A from-scratch exploration of income statement components, earnings per share, common-size analysis, and margin trends.

**Prerequisites:** Introduction to Financial Statements notebook (basic accounting equation, accrual accounting).

**Outline**
1. The multi-step income statement
2. Setup
3. Revenue, COGS, and gross profit
4. Operating expenses and operating income
5. EBITDA — uses and limitations
6. Non-operating items and net income
7. Earnings per share — basic and diluted
8. Common-size income statement (vertical analysis)
9. Horizontal analysis and margin trends
10. Comprehensive income
11. Non-recurring items and earnings quality
12. Two-company comparison
13. References

The income statement is arguably the most closely watched financial statement — it is where revenue growth, cost management, and profitability converge into a single narrative about the company's economic performance. This notebook develops the tools to read that narrative critically.

---
## 1. The Multi-Step Income Statement

The income statement reports a company's financial performance **over a period** (quarter or year). Unlike the balance sheet snapshot, it is a *flow* measure — like a video rather than a photograph.

### Single-step vs multi-step

There are two common formats:

**Single-step:** groups all revenues together and all expenses together, then computes one subtraction:

$$\text{Net Income} = \text{Total Revenues} - \text{Total Expenses}$$

**Multi-step:** presents several intermediate subtotals, each representing a different *layer* of profitability:

$$
\begin{aligned}
&\text{Revenue} \\
-\;& \text{Cost of Goods Sold (COGS)} \\
=\;& \textbf{Gross Profit} \\
-\;& \text{Selling, General \& Administrative (SG\&A)} \\
-\;& \text{Research \& Development (R\&D)} \\
-\;& \text{Depreciation \& Amortisation (D\&A)} \\
=\;& \textbf{Operating Income (EBIT)} \\
\pm\;& \text{Non-operating items (interest, FX, gains/losses)} \\
=\;& \textbf{Earnings Before Tax (EBT)} \\
-\;& \text{Income Tax Expense} \\
=\;& \textbf{Net Income}
\end{aligned}
$$

> **Key Concept:** Each subtotal answers a different analytical question: gross profit measures *production efficiency*, operating income measures *business operations*, and net income measures the *total return to shareholders*. Analysts should examine trends at every level, not just the bottom line.

> **CFA Exam Tip:** The multi-step format is strongly preferred for analysis. The CFA curriculum treats the intermediate margins (gross, operating, net) as key analytical outputs. When the exam says "income statement," assume multi-step unless stated otherwise.

### Nature of expense vs function of expense

IFRS allows companies to classify expenses by either **nature** (raw materials, employee costs, depreciation) or **function** (COGS, SG&A, R&D). US GAAP requires classification by function only.

| By Nature | By Function |
|-----------|-------------|
| Raw materials \$40M | COGS \$55M |
| Employee costs \$30M | SG&A \$25M |
| Depreciation \$10M | R&D \$15M |
| Other \$15M | D&A \$15M |

> **Common Mistake:** If a company reports by nature (common in Europe), you cannot directly compute gross profit because COGS is not broken out. You must reclassify — or accept that gross margin analysis may not be possible for that company.

### Interpreting the multi-step format — a framework

When reading any income statement, work through three questions in order:

**Question 1: Is the company growing?**
Compare revenue to the prior year. If revenue is declining, no amount of cost control can sustain profitability indefinitely.

**Question 2: Is the growth profitable?**
Check the margin progression: gross → operating → net. Healthy growth shows stable or improving margins; unhealthy growth shows expanding revenue with compressing margins (the company is "buying" growth by cutting prices or spending heavily).

**Question 3: Are the earnings sustainable?**
Look for non-recurring items, unusual gains/losses, and the gap between net income and operating income. Large below-the-line items suggest that reported earnings may not be repeatable.

> **Key Concept:** This three-question framework maps directly to the CFA curriculum's emphasis on **earnings quality**. Revenue growth addresses the top line, margin analysis addresses operational efficiency, and sustainability assessment addresses the gap between reported earnings and economic reality.

### Income statement vs cash flow statement — the critical distinction

The income statement measures **economic profit** (revenue earned minus expenses incurred), while the cash flow statement measures **cash generated** (cash received minus cash paid). The two can diverge significantly:

| Income Statement | Cash Flow Statement |
|-----------------|---------------------|
| Records revenue when *earned* | Records cash when *received* |
| Records expenses when *incurred* | Records cash when *paid* |
| Includes non-cash items (depreciation, amortisation) | Excludes non-cash items |
| Subject to estimates and judgment | Based on actual cash transactions |

A profitable company can go bankrupt if it runs out of cash (the income statement says "healthy" but the bank account says "empty"). This is why analysts never rely on the income statement alone — it must be cross-referenced with the cash flow statement, which is the subject of the next notebook in this series.

### Real-world example: Apple's income statement structure

To ground the theory, consider how Apple Inc. structures its income statement (simplified):

| Line Item | 2024 ($B) | Common-Size |
|-----------|-----------|-------------|
| Net Revenue | 391 | 100.0% |
| Cost of Sales | 210 | 53.7% |
| **Gross Margin** | **181** | **46.3%** |
| R&D Expense | 30 | 7.7% |
| SG&A Expense | 25 | 6.4% |
| **Operating Income** | **126** | **32.2%** |
| Other Income/(Expense) | 0 | 0.0% |
| **Income Before Tax** | **126** | **32.2%** |
| Income Tax Expense | 30 | 7.7% |
| **Net Income** | **96** | **24.5%** |

Apple's 46% gross margin reflects the premium pricing power of the iPhone ecosystem, and its 25% net margin is exceptional for a hardware company — achieved through massive scale (\$391B revenue spreading fixed costs thin) and a services business with near-software-like margins.

> **Key Concept:** Reading real income statements alongside the CFA curriculum makes the concepts concrete. As you progress through this series, practice pulling actual 10-K filings from SEC EDGAR and performing the same analyses you learn here.

---
## 2. Setup

This notebook builds analytical skills progressively: we start with individual measures (gross profit, operating income), then introduce comparative frameworks (common-size, horizontal analysis), and conclude with a two-company comparison that ties everything together.

All examples use synthetic data for two companies — AlphaTech Corp (hardware manufacturer) and BetaService Inc (software company) — chosen to illustrate how different business models produce dramatically different income statement profiles.

In [ ]:
%matplotlib inline
import numpy as np
from scipy import stats, optimize, linalg
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
rng = np.random.default_rng(SEED)

ATOL = 1e-10
RTOL = 1e-6

PRIMARY   = 'steelblue'
SECONDARY = 'coral'
TERTIARY  = 'seagreen'
ACCENT    = 'gold'
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})

We will work with a synthetic five-year income statement for **AlphaTech Corp**, a mid-cap technology manufacturer. The numbers are designed to exhibit realistic patterns — expanding revenue with compressing margins — that motivate each analytical technique.

AlphaTech manufactures industrial sensors and control systems. It has been growing revenue steadily through market expansion, but faces increasing competition from low-cost Asian manufacturers (driving up the need for R&D) and rising input costs (compressing gross margins). This is a common trajectory for maturing technology hardware companies and creates a rich dataset for income statement analysis.

In [ ]:
# ── AlphaTech Corp: 5-year income statement data ($ millions) ──

years = np.array([2020, 2021, 2022, 2023, 2024])

revenue       = np.array([500.0, 580.0, 650.0, 710.0, 760.0])
cogs          = np.array([200.0, 240.0, 280.0, 320.0, 355.0])
sga           = np.array([ 80.0,  90.0, 100.0, 108.0, 118.0])
rd            = np.array([ 50.0,  58.0,  65.0,  72.0,  80.0])
da            = np.array([ 25.0,  28.0,  32.0,  35.0,  38.0])
interest_exp  = np.array([ 10.0,  10.0,  12.0,  15.0,  18.0])
other_income  = np.array([  2.0,   3.0,   1.0,   4.0,   2.0])
tax_rate      = 0.25

# Derived measures
gross_profit  = revenue - cogs
opex_total    = sga + rd + da
operating_inc = gross_profit - opex_total  # EBIT
ebt           = operating_inc - interest_exp + other_income
tax_expense   = ebt * tax_rate
net_income    = ebt - tax_expense
ebitda        = operating_inc + da

print("AlphaTech Corp — Income Statement Summary ($M)")
print("=" * 65)
for i, y in enumerate(years):
    print(f"  {y}: Rev={revenue[i]:6.0f}  GP={gross_profit[i]:6.0f}  "
          f"EBIT={operating_inc[i]:6.0f}  NI={net_income[i]:6.0f}")

---
## 3. Revenue, COGS, and Gross Profit

### Revenue (top line)

Revenue — also called *sales* or *turnover* — is the starting point of the income statement. Under IFRS 15 / ASC 606, revenue is recognised when a **performance obligation** is satisfied.

Key distinctions:

* **Gross vs net revenue:** A retailer selling a \$100 product for a brand may only earn a \$15 commission. If they report \$100, that is *gross* revenue; if they report \$15, that is *net* revenue. The CFA curriculum emphasises checking whether a company is the *principal* (gross) or *agent* (net).

* **Organic vs inorganic growth:** Revenue may increase because of genuine demand (organic) or because the company acquired another business (inorganic). Analysts decompose revenue growth to assess underlying quality.

### Cost of Goods Sold (COGS)

COGS represents the **direct costs** of producing the goods or services that were sold — raw materials, direct labour, and manufacturing overhead.

* Inventory accounting methods (FIFO, LIFO, weighted average) directly affect COGS — covered in detail in the Balance Sheet & Working Capital notebook.
* A rising COGS-to-revenue ratio signals deteriorating production efficiency or input cost inflation.

### Gross Profit and Gross Margin

$$
\text{Gross Profit} = \text{Revenue} - \text{COGS}
$$

$$
\text{Gross Margin} = \frac{\text{Gross Profit}}{\text{Revenue}} \times 100\%
$$

> **Key Concept:** Gross margin is the *first filter* for assessing a business model. Software companies typically have 70-80% gross margins (low marginal cost per unit); grocery retailers may have 25-30% (thin margins on high volume). A declining gross margin is often an early warning of competitive pressure or cost inflation.

> **CFA Exam Tip:** When comparing companies across borders, check whether they classify costs by function or nature. A European company reporting by nature may include labour in a single line rather than splitting it between COGS and SG&A, making gross margin comparison misleading.

### Gross margin benchmarks by industry

Understanding typical gross margins helps analysts quickly assess whether a company's margins are reasonable:

| Industry | Typical Gross Margin | Why |
|----------|---------------------|-----|
| Software/SaaS | 70-85% | Near-zero marginal cost per user |
| Pharmaceuticals | 65-80% | Patent protection enables premium pricing |
| Luxury goods | 60-70% | Brand premium over material cost |
| Industrial manufacturing | 30-50% | Significant raw material and labour costs |
| Grocery retail | 25-30% | Thin margins, high volume |
| Airlines | 35-45% | Fuel costs dominate, intense competition |

These ranges vary by company within each industry, but a gross margin far outside the range for its peers warrants investigation.

> **Key Concept:** Gross margin is the purest measure of **pricing power** and **production efficiency**. A company with a 75% gross margin is capturing significantly more value per dollar of revenue than one with a 30% margin. Over time, companies with high and stable gross margins tend to have stronger competitive moats — they can either charge premium prices (brand, patents) or produce at lower costs (scale, technology, process efficiency).

### Revenue quality — beyond the top line number

Not all revenue is created equal. Analysts assess **revenue quality** along several dimensions:

1. **Organic vs inorganic:** Revenue from an acquisition may inflate growth temporarily but says nothing about the core business's momentum. Look for "organic growth" or "same-store sales" disclosures.

2. **Recurring vs one-time:** A software company with 90% recurring subscription revenue has far more predictable cash flows than a consulting firm that must win new contracts each quarter. Recurring revenue commands a higher valuation multiple.

3. **Diversification:** Revenue concentrated in a single customer, product, or geography is risky. SEC rules require disclosure when a single customer exceeds 10% of revenue.

4. **Channel health:** Revenue generated by "channel stuffing" (pushing excess inventory onto distributors) appears in the current quarter but often reverses in the next. Watch for receivables growing faster than revenue — a classic warning sign.

> **Key Concept:** Revenue is the single most scrutinised line on the income statement because it is both the most important and the most susceptible to manipulation. The SEC has consistently identified revenue recognition fraud as the most common type of financial statement fraud. Analysts must therefore evaluate not just the *level* of revenue but its *quality* — is it real, repeatable, and collected in cash?

### The gross margin trend as an early warning system

Gross margin is often the first metric to signal trouble because it sits closest to the revenue line and is least affected by management's discretionary spending decisions. Consider the following patterns:

* **Stable gross margin + declining revenue** → The company is maintaining pricing discipline but losing market share. May need to invest more in marketing/sales.
* **Declining gross margin + growing revenue** → The company is "buying" growth by cutting prices or absorbing cost increases. This is AlphaTech's situation — and it is unsustainable long-term.
* **Improving gross margin + growing revenue** → The ideal scenario: the company has pricing power and is growing. Often seen in companies with strong brands or network effects.
* **Declining gross margin + declining revenue** → The most dangerous pattern: the company is losing both volume and pricing power. May indicate a fundamentally deteriorating competitive position.

In [ ]:
# ── Gross profit and gross margin analysis ──

gross_margin = gross_profit / revenue * 100

fig, ax1 = plt.subplots(figsize=(10, 6))

bar_width = 0.35
x = np.arange(len(years))
ax1.bar(x - bar_width/2, revenue, bar_width, label='Revenue', color=PRIMARY, alpha=0.8)
ax1.bar(x + bar_width/2, gross_profit, bar_width, label='Gross Profit', color=TERTIARY, alpha=0.8)
ax1.set_xlabel('Year')
ax1.set_ylabel('$ Millions')
ax1.set_xticks(x)
ax1.set_xticklabels(years)
ax1.legend(loc='upper left')

ax2 = ax1.twinx()
ax2.plot(x, gross_margin, 'o-', color=SECONDARY, linewidth=2, markersize=8, label='Gross Margin %')
ax2.set_ylabel('Gross Margin (%)')
ax2.legend(loc='upper right')

ax1.set_title('AlphaTech Corp — Revenue, Gross Profit & Gross Margin')
plt.tight_layout()
plt.show()

print("Gross Margin Trend:")
for y, gm in zip(years, gross_margin):
    print(f"  {y}: {gm:.1f}%")
print(f"\nGross margin declined from {gross_margin[0]:.1f}% to {gross_margin[-1]:.1f}%")
print("→ COGS is growing faster than revenue (input cost pressure or pricing erosion)")

The chart reveals a clear trend: while both revenue and gross profit are growing in absolute terms, **gross margin is declining** — from 60.0% to 53.3%. This tells us that AlphaTech's cost of production is rising faster than its pricing power, a common pattern for maturing technology manufacturers facing competition.

### What drives gross margin changes?

Gross margin compression can stem from several sources, each with different implications:

| Cause | Signal | Analyst Response |
|-------|--------|-----------------|
| **Input cost inflation** | Commodity prices rising | Check if company can pass costs through to customers |
| **Pricing pressure** | Competitors offering lower prices | Assess competitive moat and differentiation |
| **Product mix shift** | Selling more low-margin products | Evaluate whether the mix shift is strategic |
| **Manufacturing inefficiency** | Higher waste, lower yields | Investigate operational metrics |
| **Currency effects** | Unfavourable FX on imported inputs | Assess hedging strategy |

> **Common Mistake:** A single year of gross margin decline may be noise (a one-time input cost spike). The trend over 3-5 years is what matters. AlphaTech's consistent decline from 60% to 53% over five years suggests a *structural* problem, not a temporary blip.

> **CFA Exam Tip:** When the exam presents revenue growth alongside declining margins, always identify *which* margin is declining first. If gross margin is stable but operating margin is falling, the problem is below the gross profit line (in SG&A or R&D). This distinction drives very different analytical conclusions.

---
## 4. Operating Expenses and Operating Income

Below gross profit, we deduct **operating expenses** — the costs of running the business beyond direct production:

* **SG&A (Selling, General & Administrative):** sales commissions, marketing, executive salaries, office rent, legal fees.
* **R&D (Research & Development):** costs of developing new products or technologies. Under US GAAP, R&D is *always* expensed; under IFRS, *development* costs meeting specific criteria may be capitalised.
* **D&A (Depreciation & Amortisation):** systematic allocation of the cost of tangible (depreciation) and intangible (amortisation) assets over their useful lives.

$$
\text{Operating Income (EBIT)} = \text{Gross Profit} - \text{SG\&A} - \text{R\&D} - \text{D\&A}
$$

$$
\text{Operating Margin} = \frac{\text{EBIT}}{\text{Revenue}} \times 100\%
$$

> **Key Concept:** Operating income isolates the profitability of the *core business* — before the effects of financing decisions (interest) and tax jurisdiction. It is the most important measure for comparing companies with different capital structures.

> **Common Mistake:** D&A is a *non-cash* expense — no cash leaves the company when depreciation is recorded. However, it represents real economic cost (the wear and tear on assets that will eventually need replacement). Ignoring D&A overstates true operating profitability.

### R&D capitalisation — IFRS vs US GAAP

One of the most significant IFRS/GAAP differences affects operating expenses directly:

* **US GAAP:** All research and development costs are expensed as incurred (with narrow exceptions for software development costs under ASC 350-40).
* **IFRS (IAS 38):** *Research* costs are expensed, but *development* costs can be **capitalised** as intangible assets if six criteria are met (technical feasibility, intention to complete, ability to use or sell, probable future economic benefits, adequate resources, ability to measure costs reliably).

The implications for analysis are significant:
* An IFRS company that capitalises development costs will report **higher operating income** (less R&D expense) and **higher assets** (the capitalised amount appears as an intangible).
* The cash flow impact is identical — the cash is spent regardless. But it shifts from operating (expense) to investing (capitalised asset) in the cash flow statement.

> **CFA Exam Tip:** When comparing a US GAAP company (all R&D expensed) with an IFRS company (some development capitalised), you must adjust one to match the other. The most common approach is to *expense* the IFRS company's capitalised development costs — add back the capitalised amount to R&D expense and subtract it from intangible assets.

### SG&A — a window into management efficiency

SG&A is often the most controllable expense category, and its trend reveals management priorities:

* **SG&A declining as % of revenue** → Operating leverage is kicking in; the company is growing efficiently.
* **SG&A increasing as % of revenue** → The company may be over-hiring, expanding into expensive markets, or experiencing diseconomies of scale.
* **SG&A cut sharply** → Management may be sacrificing future growth (cutting marketing, firing salespeople) to boost current profits — a short-term fix with long-term consequences.

In [ ]:
# ── Operating expense breakdown and operating margin ──

operating_margin = operating_inc / revenue * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: stacked operating expense breakdown
x = np.arange(len(years))
ax1.bar(x, cogs, label='COGS', color=PRIMARY)
ax1.bar(x, sga, bottom=cogs, label='SG&A', color=SECONDARY)
ax1.bar(x, rd, bottom=cogs+sga, label='R&D', color=TERTIARY)
ax1.bar(x, da, bottom=cogs+sga+rd, label='D&A', color=ACCENT)
ax1.set_xticks(x)
ax1.set_xticklabels(years)
ax1.set_ylabel('$ Millions')
ax1.set_title('Cost Structure Breakdown')
ax1.legend()

# Right: margin waterfall for latest year
margins = {
    'Gross
Margin': gross_margin[-1],
    'After
SG&A': (gross_profit[-1] - sga[-1]) / revenue[-1] * 100,
    'After
R&D': (gross_profit[-1] - sga[-1] - rd[-1]) / revenue[-1] * 100,
    'Operating
Margin': operating_margin[-1]
}
labels = list(margins.keys())
vals = list(margins.values())
colors = [PRIMARY, SECONDARY, TERTIARY, ACCENT]
ax2.bar(labels, vals, color=colors, edgecolor='white', width=0.5)
for i, v in enumerate(vals):
    ax2.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=11, fontweight='bold')
ax2.set_ylabel('Margin (%)')
ax2.set_title(f'Margin Erosion — {years[-1]}')

plt.tight_layout()
plt.show()

print(f"\n2024 Operating Margin: {operating_margin[-1]:.1f}%")
print("Each operating expense category reduces the margin further:")
for label, val in margins.items():
    print(f"  {label.replace(chr(10), ' ')}: {val:.1f}%")

The stacked bar chart shows that COGS is the largest cost component and is growing rapidly. The margin erosion chart for 2024 illustrates how each category of operating expense chips away at gross margin, leaving an operating margin of approximately 22%.

### Operating leverage — the magnification effect

**Operating leverage** describes how sensitive operating income is to changes in revenue. Companies with high fixed costs (rent, salaries, depreciation) have high operating leverage — a small revenue increase produces a large profit increase (and vice versa).

$$\text{Degree of Operating Leverage (DOL)} = \frac{\%\Delta \text{Operating Income}}{\%\Delta \text{Revenue}}$$

For AlphaTech, the increasing share of fixed costs (R&D, depreciation) relative to revenue means operating leverage is rising. This creates an asymmetric risk profile: in good years, profits grow faster than revenue; in a downturn, profits collapse faster than revenue.

> **Key Concept:** When analysing operating expenses, distinguish between **discretionary** costs (R&D, marketing — can be cut quickly) and **committed** costs (depreciation, long-term leases — cannot be cut). A company that has been cutting R&D to maintain operating margins may be sacrificing future growth for current profitability — a trade-off that won't show up in the income statement until years later.

### Depreciation and amortisation — the hidden cost

D&A deserves special attention because it is simultaneously:
* A **non-cash expense** (no money leaves the company when depreciation is recorded)
* A **real economic cost** (the asset is genuinely wearing out and will need replacement)

Companies sometimes argue that D&A should be ignored because "it's non-cash." This is dangerously misleading. A trucking company's fleet depreciates not because accountants say so, but because the trucks physically deteriorate. Ignoring depreciation overstates the company's ability to distribute cash to shareholders — some of that cash must be reinvested to maintain capacity.

> **CFA Exam Tip:** The ratio of **capex to depreciation** is a key diagnostic. If capex consistently exceeds depreciation, the company is investing in growth (expanding capacity). If capex is below depreciation, the company may be harvesting — maintaining capacity just enough to keep the current business running, or even shrinking. The CFA curriculum views sustained capex < depreciation as a warning sign for long-term competitiveness.

---
## 5. EBITDA — Uses and Limitations

**EBITDA** (Earnings Before Interest, Taxes, Depreciation & Amortisation) is one of the most widely cited profitability metrics in finance, yet it is also one of the most debated.

$$
\text{EBITDA} = \text{EBIT} + \text{D\&A} = \text{Net Income} + \text{Tax} + \text{Interest} + \text{D\&A}
$$

### Why practitioners use EBITDA

1. **Capital structure neutrality:** By adding back interest, EBITDA removes the effect of leverage — useful for comparing companies with different debt levels.
2. **Depreciation policy neutrality:** Different companies depreciate assets over different lives; adding back D&A removes this source of variation.
3. **Proxy for operating cash flow:** In a simplified world, EBITDA approximates cash generated by operations (before working capital changes and capex).
4. **Valuation multiple:** EV/EBITDA is the most common enterprise value multiple, widely used in M&A and credit analysis.

### The case against EBITDA

> **Key Concept:** Warren Buffett famously dismissed EBITDA: *"Does management think the tooth fairy pays for capital expenditures?"*

The criticisms are well-founded:

* **Ignores capital intensity:** Two companies with identical EBITDA may have wildly different capital requirements. A software company needs minimal capex; a steel manufacturer needs massive reinvestment. EBITDA treats them as equivalent.
* **Not a GAAP/IFRS measure:** Companies can define "adjusted EBITDA" however they wish — adding back stock-based compensation, restructuring charges, or other items. This makes cross-company comparison treacherous.
* **Ignores working capital:** EBITDA does not account for cash tied up in receivables and inventory.
* **Overstates cash generation:** A mature company must replace depreciating assets to maintain capacity. Ignoring depreciation pretends this cost doesn't exist.

> **CFA Exam Tip:** The CFA curriculum treats EBITDA as a useful but flawed metric. Expect exam questions testing whether you understand its limitations — especially the distinction between EBITDA and actual free cash flow.

### "Adjusted EBITDA" — the wild west of financial metrics

Many companies report "adjusted EBITDA" that adds back not just D&A but also:
* Stock-based compensation
* Restructuring charges
* Acquisition-related costs
* Litigation settlements
* Foreign exchange losses

Each add-back may be individually reasonable, but the cumulative effect can be enormous. A study by Audit Analytics found that the median difference between GAAP net income and adjusted EBITDA for S&P 500 companies was over 50%.

> **Common Mistake:** Never accept "adjusted EBITDA" without examining the adjustments. A company adding back stock-based compensation is claiming that paying employees with shares has no economic cost — but it dilutes existing shareholders, which is a very real cost. When comparing companies, either use the same adjustments for all, or use a GAAP-defined measure like operating income.

### EBIT vs EBITDA — when to use which

| Use EBIT when... | Use EBITDA when... |
|-------------------|---------------------|
| Comparing companies in the same industry | Comparing capital-intensive vs asset-light companies |
| D&A is a meaningful economic cost | D&A policies vary significantly across comparables |
| Focus is on operational profitability | Focus is on cash-generating capacity (approximate) |
| The company has stable capex needs | The company has lumpy, non-recurring capex |

> **Key Concept:** Neither EBIT nor EBITDA is "better" in all situations. The choice depends on what aspect of profitability you are trying to isolate. For most analytical purposes, **free cash flow** (covered in the Cash Flow notebook) is superior to both because it accounts for actual capital expenditure requirements rather than ignoring them entirely (EBITDA) or using a historical-cost proxy (EBIT).

In [ ]:
# ── EBITDA vs Operating Income vs Net Income ──

ebitda_margin  = ebitda / revenue * 100
net_margin     = net_income / revenue * 100

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(years, ebitda_margin, 's-', color=ACCENT, linewidth=2, markersize=8, label='EBITDA Margin')
ax.plot(years, operating_margin, 'o-', color=PRIMARY, linewidth=2, markersize=8, label='Operating Margin (EBIT)')
ax.plot(years, net_margin, '^-', color=SECONDARY, linewidth=2, markersize=8, label='Net Margin')

ax.fill_between(years, operating_margin, ebitda_margin, alpha=0.15, color=ACCENT, label='D&A effect')
ax.fill_between(years, net_margin, operating_margin, alpha=0.15, color=SECONDARY, label='Interest + Tax effect')

ax.set_xlabel('Year')
ax.set_ylabel('Margin (%)')
ax.set_title('AlphaTech Corp — Margin Comparison: EBITDA vs EBIT vs Net Income')
ax.legend(loc='lower left')
plt.tight_layout()
plt.show()

print("Margin Comparison:")
print(f"{'Year':>6} {'EBITDA%':>9} {'EBIT%':>9} {'Net%':>9} {'D&A gap':>9} {'Below-EBIT gap':>14}")
for i in range(len(years)):
    print(f"{years[i]:>6} {ebitda_margin[i]:>8.1f}% {operating_margin[i]:>8.1f}% {net_margin[i]:>8.1f}% "
          f"{ebitda_margin[i]-operating_margin[i]:>8.1f}% {operating_margin[i]-net_margin[i]:>13.1f}%")

The shaded regions quantify the gap between the three margin measures. The amber region (D&A) is relatively stable — depreciation is a predictable, scheduled expense. The coral region (interest + tax) is widening because AlphaTech has been increasing its debt load (interest expense rose from \$10M to \$18M).

### The growing interest burden

AlphaTech's interest expense nearly doubled over the five-year period. Combined with margin compression, this creates a "double squeeze" on net income: the numerator (operating profit) is growing slowly while a larger share is being absorbed by debt service.

This is visible in the widening gap between EBITDA margin and net margin — from approximately 8 percentage points in 2020 to nearly 12 in 2024. Analysts call this **financial leverage risk**: as debt increases, a fixed interest obligation consumes an increasing share of operating profits, leaving less for shareholders.

> **Key Concept:** EBITDA margin is often used to compare companies with different capital structures because it excludes interest. But this is precisely its limitation — it hides the fact that a highly leveraged company has a much smaller margin of safety. A 5% drop in revenue might be survivable for an unlevered company but could push a leveraged company into financial distress.

### Margin convergence at the bottom line

One of the most important observations from the three-margin chart is that the margins are **converging** — the gap between EBITDA margin and net margin is widening. This means an increasing share of operating profits is being absorbed by D&A, interest, and taxes before reaching shareholders.

For investors, this convergence pattern signals rising risk: as the "margin cushion" between EBITDA and net income shrinks, even a small revenue disappointment could push net income negative while EBITDA remains positive. Companies in this situation are sometimes called "EBITDA-positive but unprofitable" — a description that applies to many growth-stage companies and leveraged buyout targets.

---
## 6. Non-Operating Items and Net Income

Below operating income, we encounter items that are **not part of the core business**:

### Interest expense and interest income

Interest expense reflects the cost of debt financing. Interest income comes from cash investments. The *net interest* figure reveals the cost of the company's capital structure decisions.

$$\text{EBT} = \text{EBIT} - \text{Net Interest Expense} + \text{Other Income/Losses}$$

### Other income and losses

This category captures:
* Foreign exchange gains/losses
* Gains/losses on sale of assets
* Fair value changes on investments (if marked to market)
* Impairment charges

> **Common Mistake:** A one-time gain from selling a building can inflate net income, making the company look more profitable than its core operations warrant. Always separate recurring from non-recurring items.

### Income tax expense

$$\text{Net Income} = \text{EBT} - \text{Income Tax Expense}$$

The **effective tax rate** is:

$$\text{Effective Tax Rate} = \frac{\text{Income Tax Expense}}{\text{EBT}} \times 100\%$$

This may differ from the statutory rate due to tax credits, deferred taxes, and jurisdictional differences. A persistently low effective tax rate may indicate aggressive tax planning — or it may reflect legitimate R&D credits.

> **Key Concept:** Net income is the "bottom line" — the ultimate measure of profitability attributable to shareholders. But because it includes non-recurring items, financing effects, and tax strategies, it is often the *noisiest* profit measure. Analysts frequently prefer operating income or adjusted earnings for trend analysis.

### The effective tax rate — a diagnostic tool

The effective tax rate (ETR) is one of the most revealing analytical metrics:

$$\text{ETR} = \frac{\text{Income Tax Expense}}{\text{Earnings Before Tax}}$$

A company's ETR may differ from the statutory rate for several reasons:

| Factor | Effect on ETR | Persistence |
|--------|--------------|-------------|
| R&D tax credits | Lowers ETR | Recurring (if policy continues) |
| Foreign income at lower rates | Lowers ETR | Recurring (depends on geography) |
| Tax-exempt municipal bond income | Lowers ETR | Recurring |
| Non-deductible expenses (fines, penalties) | Raises ETR | Usually non-recurring |
| Deferred tax asset valuation allowance | Raises ETR | Signals doubt about future profitability |
| One-time repatriation taxes | Raises ETR | Non-recurring |

> **CFA Exam Tip:** A company whose ETR has been declining over time may be using increasingly aggressive tax strategies. While legal, aggressive tax planning creates risk — a change in tax law or a successful IRS challenge could cause a sudden jump in tax expense. Analysts should assess whether the current ETR is sustainable when projecting future earnings.

In [ ]:
# ── Revenue-to-net-income waterfall for 2024 ──

labels = ['Revenue', 'COGS', 'SG&A', 'R&D', 'D&A', 'Interest', 'Other Inc.', 'Tax', 'Net Income']
values = [revenue[-1], -cogs[-1], -sga[-1], -rd[-1], -da[-1], -interest_exp[-1],
          other_income[-1], -tax_expense[-1], net_income[-1]]

# Compute waterfall positions
running = np.cumsum(values[:-1])
bottoms = np.zeros(len(values))
bottoms[0] = 0
for i in range(1, len(values)-1):
    bottoms[i] = min(running[i-1], running[i-1] + values[i]) if values[i] < 0 else running[i-1]
bottoms[-1] = 0  # Net income starts from zero

heights = [abs(v) for v in values]
colors_wf = [PRIMARY if v > 0 else SECONDARY for v in values]
colors_wf[0] = PRIMARY     # Revenue
colors_wf[-1] = TERTIARY   # Net Income (subtotal)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(labels, heights, bottom=bottoms, color=colors_wf, edgecolor='white', width=0.6)
for bar, val in zip(bars, values):
    y_pos = bar.get_y() + bar.get_height() + 3
    ax.text(bar.get_x() + bar.get_width()/2, y_pos, f'${val:+.0f}M', ha='center', fontsize=9, fontweight='bold')

ax.set_ylabel('$ Millions')
ax.set_title('AlphaTech Corp 2024 — Revenue to Net Income Waterfall')
ax.axhline(0, color='grey', linewidth=0.5)
plt.tight_layout()
plt.show()

The waterfall makes it immediately clear where value is created and consumed. COGS is the largest deduction (\$355M), followed by SG&A (\$118M) and R&D (\$80M). The final green bar represents net income — what remains for shareholders after all costs, interest, and taxes.

### Reading the waterfall for analytical insights

The waterfall format is powerful because it shows the **relative magnitude** of each cost category at a glance:

* COGS consumes **46.7%** of revenue — the dominant expense for any manufacturer.
* SG&A at **15.5%** is typical for a B2B technology company (lower than consumer-facing businesses).
* R&D at **10.5%** reflects the need to continuously innovate — cutting this would be strategically dangerous.
* Tax at **6.0%** is mechanically driven by the statutory rate and pre-tax profit level.

> **CFA Exam Tip:** When given a waterfall or common-size breakdown, quickly identify the largest two or three cost categories. These are where a 1-2 percentage point change would have the most material impact on profitability. For AlphaTech, even a small improvement in COGS (e.g., from 46.7% to 45%) would boost net income by approximately \$10M — more than any realistic cut to SG&A or R&D.

### The retention ratio and sustainable growth

From the waterfall, we can also compute the **sustainable growth rate** — the rate at which the company can grow using only internal resources (retained earnings):

$$g_{\text{sustainable}} = ROE \times \text{Retention Ratio} = ROE \times \left(1 - \frac{\text{Dividends}}{\text{Net Income}}\right)$$

This formula connects the income statement (profitability and dividend policy) to the balance sheet (equity growth) to the company's future trajectory — a core concept in equity valuation that we preview here and develop fully in later notebooks.

---
## 7. Earnings Per Share — Basic and Diluted

EPS is the single most widely followed metric in equity analysis. It expresses net income on a *per share* basis, enabling comparison across companies of different sizes.

### Basic EPS

$$
\text{Basic EPS} = \frac{\text{Net Income} - \text{Preferred Dividends}}{\text{Weighted Average Common Shares Outstanding}}
$$

The weighted average accounts for shares issued or repurchased during the year:

$$
\text{WACS} = \sum_{\text{periods}} (\text{shares outstanding}_i \times \text{fraction of year}_i)
$$

### Diluted EPS

Diluted EPS assumes that all *potentially dilutive* securities (stock options, warrants, convertible bonds, convertible preferred shares) are converted into common shares. It answers: *"What would EPS be in the worst case for existing shareholders?"*

$$
\text{Diluted EPS} = \frac{\text{Net Income} - \text{Preferred Div.} + \text{Convertible Adjustments}}{\text{WACS} + \text{Dilutive Shares}}
$$

### The treasury stock method (for options and warrants)

When options are exercised, the company receives the exercise price. It is assumed to use these proceeds to buy back shares at the *average market price*:

$$
\text{Net new shares} = \text{Options exercisable} - \frac{\text{Options} \times \text{Exercise Price}}{\text{Average Market Price}}
$$

Options are dilutive only if the exercise price < market price (i.e., they are "in the money").

> **Key Concept:** The treasury stock method assumes the company uses option exercise proceeds to repurchase shares at market price. This nets out the cash inflow, showing only the *incremental* dilution.

> **CFA Exam Tip:** Anti-dilutive securities (those that would *increase* EPS) are **excluded** from the diluted calculation. Always test each potential dilutive security individually: if including it raises EPS, exclude it.

### Worked example

Consider AlphaTech in 2024:
* Net income: computed above
* Preferred dividends: \$2M
* Shares outstanding: 50M shares (all year)
* Stock options: 5M options, exercise price \$15, average market price \$25
* Convertible bonds: \$20M face, 6% coupon, convertible into 2M shares

### Weighted average shares — handling mid-year changes

When shares are issued or repurchased during the year, the denominator must be **time-weighted**:

**Example:** A company had 10 million shares outstanding on January 1. On April 1, it issued 2 million new shares. On October 1, it repurchased 1 million shares.

$$\text{WACS} = 10M \times \frac{3}{12} + 12M \times \frac{6}{12} + 11M \times \frac{3}{12} = 2.5M + 6M + 2.75M = 11.25M$$

> **Key Concept:** Stock dividends and stock splits are treated differently from new issuances. Because they give existing shareholders proportionally more shares without any new capital entering the company, they are applied **retroactively** — as if they occurred at the beginning of the earliest period presented. This ensures comparability across periods.

### The P/E ratio connection

EPS is the denominator in the most widely used valuation metric:

$$\text{P/E Ratio} = \frac{\text{Market Price per Share}}{\text{EPS}}$$

Trailing P/E uses the most recent 12 months of reported EPS; forward P/E uses analysts' consensus EPS forecast. Because EPS directly determines the P/E multiple, any of the income statement issues discussed in this notebook (margin compression, non-recurring items, EPS dilution) directly affect how the market values the stock.

### EPS and share buybacks

Share repurchases (buybacks) reduce the denominator of the EPS fraction, mechanically increasing EPS even if net income is unchanged. This is a common strategy for "managing" EPS growth:

**Example:** A company earns \$100M with 50M shares → EPS = \$2.00. It borrows \$200M to buy back 10M shares. Next year it earns \$98M (slight decline due to interest expense on the debt) with 40M shares → EPS = \$2.45.

EPS grew 22.5% even though *net income declined*. The entire "growth" came from financial engineering, not operational improvement.

> **Common Mistake:** Rising EPS does not necessarily mean the business is improving. Always check whether EPS growth is driven by *numerator growth* (higher net income — genuine improvement) or *denominator shrinkage* (share buybacks — financial engineering). The CFA curriculum emphasises this distinction and may test it explicitly.

In [ ]:
# ── EPS Calculation: Basic and Diluted ──

ni_2024 = net_income[-1]   # $ millions
pref_div = 2.0             # $ millions
shares_outstanding = 50.0  # millions

# Basic EPS
basic_eps = (ni_2024 - pref_div) / shares_outstanding
print(f"Basic EPS = (${ni_2024:.1f}M - ${pref_div:.1f}M) / {shares_outstanding:.0f}M shares = ${basic_eps:.4f}")

# ── Treasury Stock Method for options ──
options = 5.0        # millions of options
exercise_price = 15  # per share
market_price = 25    # average market price per share

# Shares from option exercise
shares_from_options = options
# Shares repurchased with exercise proceeds
proceeds = options * exercise_price  # $75M
shares_repurchased = proceeds / market_price  # 3M shares
net_new_shares_options = shares_from_options - shares_repurchased

print(f"\nTreasury Stock Method:")
print(f"  Options exercised:     {shares_from_options:.0f}M shares")
print(f"  Exercise proceeds:     ${proceeds:.0f}M (= {options:.0f}M × ${exercise_price})")
print(f"  Shares repurchased:    {shares_repurchased:.0f}M shares (= ${proceeds:.0f}M / ${market_price})")
print(f"  Net new shares:        {net_new_shares_options:.0f}M shares")

# ── Convertible bonds ──
convert_face = 20.0     # $ millions
convert_coupon = 0.06   # 6% annual
convert_shares = 2.0    # millions of new shares
tax_rate_val = 0.25

# If converted: no interest expense saved (after tax)
interest_saved = convert_face * convert_coupon * (1 - tax_rate_val)

print(f"\nConvertible Bond Adjustment:")
print(f"  Interest saved (after-tax): ${interest_saved:.2f}M")
print(f"  Additional shares:          {convert_shares:.0f}M")

# ── Diluted EPS (check each security) ──
# Test options: add shares, no income adjustment
# Test convertibles: add income and shares

# Step 1: Options are dilutive if exercise price < market price
print(f"\nDilution Tests:")
print(f"  Options: Exercise ${exercise_price} < Market ${market_price} → DILUTIVE")

# Step 2: Check convertible — per-share impact
convert_per_share = interest_saved / convert_shares
print(f"  Convertible: ${interest_saved:.2f}M / {convert_shares:.0f}M = ${convert_per_share:.4f} per share")

# Start with basic, add options first (no income effect)
diluted_ni = ni_2024 - pref_div + interest_saved
diluted_shares = shares_outstanding + net_new_shares_options + convert_shares

diluted_eps = diluted_ni / diluted_shares
print(f"  Basic EPS for comparison: ${basic_eps:.4f}")
print(f"  Convertible per-share impact (${convert_per_share:.4f}) < Basic EPS (${basic_eps:.4f}) → DILUTIVE")

print(f"\nDiluted EPS = (${ni_2024:.1f} - ${pref_div:.1f} + ${interest_saved:.2f}) / ({shares_outstanding:.0f} + {net_new_shares_options:.0f} + {convert_shares:.0f})")
print(f"Diluted EPS = ${diluted_ni:.2f}M / {diluted_shares:.0f}M = ${diluted_eps:.4f}")
print(f"\nDilution impact: ${basic_eps:.4f} → ${diluted_eps:.4f} ({(diluted_eps/basic_eps - 1)*100:.1f}%)")

The diluted EPS is lower than basic EPS, quantifying the potential impact on existing shareholders if all in-the-money options are exercised and convertible bonds are converted.

> **Common Mistake:** Students often forget that convertible bond conversion *increases* the numerator (adding back after-tax interest saved) as well as the denominator (adding new shares). Both adjustments must be made simultaneously.

### Why the dilution gap matters

The gap between basic and diluted EPS is a measure of **potential shareholder dilution**. A large gap indicates that the company has significant outstanding options, warrants, or convertible securities that could reduce each shareholder's claim on earnings.

Investors and analysts typically focus on diluted EPS because it represents the "worst case" for per-share earnings. Companies with large employee stock option programs (common in technology) often have meaningful dilution — sometimes 5-10% or more.

### Anti-dilutive securities

Not all convertible securities are included in diluted EPS. If including a security would *increase* EPS (i.e., the per-share earnings impact of conversion exceeds the current diluted EPS), it is **anti-dilutive** and must be excluded.

The most common example: out-of-the-money stock options (exercise price > market price). These would never be rationally exercised, so they create no dilution.

> **CFA Exam Tip:** The exam may present a scenario with multiple potentially dilutive securities and ask you to determine which should be included. The correct approach is to rank securities from most dilutive to least dilutive, adding them one at a time. Stop adding when the next security would be anti-dilutive.

---
## 8. Common-Size Income Statement (Vertical Analysis)

A **common-size** income statement expresses every line item as a **percentage of revenue**. This removes the effect of company size, enabling direct comparison between:
* The **same company** across time periods (trend analysis)
* **Different companies** in the same period (cross-sectional analysis)

$$
\text{Common-size } X = \frac{X}{\text{Revenue}} \times 100\%
$$

> **Key Concept:** Common-size analysis is the single most powerful tool for detecting margin trends. A company may report record revenue, but if its common-size COGS is rising every year, profitability is actually deteriorating.

> **CFA Exam Tip:** The exam frequently presents a common-size income statement and asks you to identify trends or compare two companies. Be able to read these tables quickly and spot the key changes.

### Vertical vs horizontal analysis

These two complementary techniques answer different questions:

| Technique | Question Answered | Base |
|-----------|-------------------|------|
| **Vertical (common-size)** | What fraction of revenue is each item? | Revenue in the *same* year |
| **Horizontal (indexed)** | How fast is each item growing? | Same item in the *base* year |

Used together, they provide a complete picture. Common-size shows the current cost structure; horizontal analysis shows how it is changing. A cost that is small in common-size terms but growing rapidly in horizontal terms may become a problem in the future.

In [ ]:
# ── Common-size income statement (all items as % of revenue) ──

def common_size_table(years, revenue, cogs, sga, rd, da, interest_exp, other_income, tax_expense, net_income):
    """Print a common-size income statement table."""
    items = {
        'Revenue':       revenue,
        'COGS':          cogs,
        'Gross Profit':  revenue - cogs,
        'SG&A':          sga,
        'R&D':           rd,
        'D&A':           da,
        'Operating Inc.': revenue - cogs - sga - rd - da,
        'Interest Exp.': interest_exp,
        'Other Income':  other_income,
        'EBT':           revenue - cogs - sga - rd - da - interest_exp + other_income,
        'Tax Expense':   tax_expense,
        'Net Income':    net_income,
    }

    # Header
    print(f"{'':>18}", end='')
    for y in years:
        print(f"  {y:>8}", end='')
    print()
    print("─" * (18 + 10 * len(years)))

    for name, vals in items.items():
        pcts = vals / revenue * 100
        print(f"{name:>18}", end='')
        for p in pcts:
            print(f"  {p:>7.1f}%", end='')
        print()
        if name in ['Gross Profit', 'Operating Inc.', 'EBT']:
            print("─" * (18 + 10 * len(years)))

common_size_table(years, revenue, cogs, sga, rd, da, interest_exp, other_income, tax_expense, net_income)

The common-size table above makes trend identification effortless. Key observations:

* **COGS** rose from 40.0% to 46.7% of revenue — the single largest driver of margin compression.
* **R&D** increased from 10.0% to 10.5% — a modest but intentional increase as AlphaTech invests in next-generation products.
* **Net income** fell from 16.9% to 13.2% — a 3.7 percentage point decline that compounds into significant absolute dollar differences at AlphaTech's scale.

The visualisation below presents the same data as a stacked area chart, making it easier to see how the composition of each revenue dollar has shifted over time.

In [ ]:
# ── Visualise common-size trends ──

fig, ax = plt.subplots(figsize=(12, 7))

# Stacked area chart of costs as % of revenue
cogs_pct = cogs / revenue * 100
sga_pct  = sga / revenue * 100
rd_pct   = rd / revenue * 100
da_pct   = da / revenue * 100
int_pct  = interest_exp / revenue * 100
tax_pct  = tax_expense / revenue * 100
ni_pct   = net_income / revenue * 100

ax.stackplot(years, cogs_pct, sga_pct, rd_pct, da_pct, int_pct, tax_pct, ni_pct,
             labels=['COGS', 'SG&A', 'R&D', 'D&A', 'Interest', 'Tax', 'Net Income'],
             colors=[PRIMARY, SECONDARY, TERTIARY, ACCENT, '#9B59B6', '#95A5A6', '#2ECC71'],
             alpha=0.8)

ax.set_xlabel('Year')
ax.set_ylabel('% of Revenue')
ax.set_title('AlphaTech Corp — Common-Size Income Statement (Stacked)')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

The stacked chart makes margin compression immediately visible. The blue COGS band is expanding year over year, squeezing the green net income band at the top. While R&D spending is also growing as a percentage of revenue, the dominant driver of margin erosion is production costs.

### Using common-size analysis in practice

Common-size analysis is most valuable in two contexts:

1. **Time-series (same company, multiple years):** Track how cost structure evolves. Are margins expanding or contracting? Which cost categories are responsible?

2. **Cross-sectional (multiple companies, same year):** Compare business models. A software company at 80% gross margin vs a retailer at 25% reflects fundamentally different economics, not "better" or "worse" management.

> **Key Concept:** Common-size analysis is the first step in almost every professional financial analysis. It transforms raw dollar figures into percentages that can be compared across companies of vastly different sizes and across time periods with different price levels. Master this technique and you have a tool that works for any company in any industry.

### Limitations of common-size analysis

While powerful, common-size analysis has blind spots:

1. **Scale effects disappear** — a company earning 15% net margin on \$10 billion of revenue generates \$1.5 billion in profit. The same margin on \$100 million of revenue generates only \$15 million. Common-size analysis cannot distinguish between these.

2. **Accounting policy differences** — two companies with identical operations but different depreciation methods, inventory methods, or revenue recognition policies will show different common-size profiles.

3. **Business cycle sensitivity** — margins fluctuate with the economic cycle. Comparing a company's peak-cycle margins to a peer's trough-cycle margins is misleading.

4. **One-time items** — a large restructuring charge in one year will distort the common-size percentages for that period, making trend analysis unreliable unless the item is excluded.

> **Key Concept:** Common-size analysis should be the *starting point* of income statement analysis, not the conclusion. It identifies where to focus deeper investigation — which cost categories are changing, which margins are compressing — but the analyst must then determine *why* by reading footnotes, management commentary, and industry reports.

---
## 9. Horizontal Analysis and Margin Trends

While common-size analysis compares items to revenue *within each year* (vertical), **horizontal analysis** tracks each item's growth *across years*, using a base year:

$$
\text{Index}_{t} = \frac{\text{Value}_{t}}{\text{Value}_{\text{base}}} \times 100
$$

This reveals which items are growing faster or slower than revenue — the key to understanding margin trends.

> **Key Concept:** If COGS grows at 15% per year but revenue grows at only 10%, gross margin will inevitably compress. Horizontal analysis makes these divergent growth rates explicit.

### The base year problem

The choice of base year can significantly affect the interpretation:

* If the base year was abnormally **good** (e.g., a cyclical peak), all subsequent years look poor by comparison.
* If the base year was abnormally **bad** (e.g., a recession trough), all subsequent years show impressive growth that may merely reflect recovery, not genuine improvement.

> **Common Mistake:** When presenting horizontal analysis, always note whether the base year was representative. If Year 1 was a recession year, a 50% revenue increase by Year 3 may simply mean the company has returned to pre-recession levels rather than achieving genuine growth.

### Trend extrapolation — use with caution

It is tempting to extrapolate observed trends forward (e.g., "if COGS continues growing at 15% CAGR..."). But financial trends are rarely linear for long:

* Margins have natural floors and ceilings (you cannot have negative COGS or >100% margin).
* Companies adapt: management responds to margin pressure by cutting costs, raising prices, or pivoting strategy.
* Competitive dynamics shift: new entrants, regulatory changes, and technological disruption can all bend or reverse trends.

Use trend analysis to *identify problems early*, not to predict the future mechanically.

In [ ]:
# ── Horizontal analysis (2020 = 100) and margin trends ──

base = 0  # 2020 as base year
items_ha = {
    'Revenue':    revenue,
    'COGS':       cogs,
    'SG&A':       sga,
    'R&D':        rd,
    'Net Income': net_income,
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: indexed growth
colors_ha = [PRIMARY, SECONDARY, TERTIARY, ACCENT, 'black']
for (name, vals), color in zip(items_ha.items(), colors_ha):
    indexed = vals / vals[base] * 100
    ax1.plot(years, indexed, 'o-', color=color, linewidth=2, label=name)

ax1.axhline(100, color='grey', linewidth=0.5, linestyle='--')
ax1.set_xlabel('Year')
ax1.set_ylabel('Index (2020 = 100)')
ax1.set_title('Horizontal Analysis — Growth Indices')
ax1.legend()

# Right: all three margins
gm = gross_profit / revenue * 100
om = operating_inc / revenue * 100
nm = net_income / revenue * 100

ax2.plot(years, gm, 's-', color=PRIMARY, linewidth=2, markersize=8, label='Gross Margin')
ax2.plot(years, om, 'o-', color=SECONDARY, linewidth=2, markersize=8, label='Operating Margin')
ax2.plot(years, nm, '^-', color=TERTIARY, linewidth=2, markersize=8, label='Net Margin')
ax2.set_xlabel('Year')
ax2.set_ylabel('Margin (%)')
ax2.set_title('Margin Trends')
ax2.legend()

plt.tight_layout()
plt.show()

print("Growth rates (CAGR 2020-2024):")
for name, vals in items_ha.items():
    cagr = (vals[-1] / vals[0]) ** (1/4) - 1
    print(f"  {name:>12}: {cagr*100:.1f}%")

The left chart confirms the diagnosis: COGS is growing faster than revenue (indexed COGS line is steeper), driving margin compression. R&D is also outpacing revenue growth. The right chart shows all three margins declining in parallel, with net margin suffering the most due to rising interest expense layered on top of operating margin compression.

### CAGR decomposition — the most important number

The compound annual growth rates (CAGRs) reveal the speed of divergence:

If COGS grows at a CAGR of ~15% while revenue grows at ~11%, the gap compounds: after 5 years, COGS has grown ~77% vs revenue's ~52%. This 25-percentage-point divergence is what transforms a 60% gross margin into a 53% gross margin.

> **CFA Exam Tip:** Horizontal analysis is especially powerful for detecting **cost creep** — gradual increases in expense ratios that are easy to miss in any single year but compound into significant margin erosion over time. Always compute indexed growth rates alongside common-size percentages; together they provide both the "what" (common-size) and the "how fast" (horizontal) dimensions of the story.

### The DuPont connection

The margin trends visible in horizontal analysis feed directly into the **DuPont decomposition** (covered in the Ratio Analysis & DuPont notebook). If net margin is declining, it will drag down ROE unless offset by higher asset turnover or increased leverage. Understanding margin trends is therefore the first step in understanding what is driving changes in shareholder returns.

> **Key Concept:** Income statement trends rarely exist in isolation. Declining margins often coincide with balance sheet changes (inventory buildup, receivables growth) and cash flow changes (widening gap between NI and CFO). The most powerful analysis connects trends across all three statements — a skill developed over the remaining notebooks in this series.

### Using horizontal analysis to set expectations

By establishing a baseline growth rate for each line item, horizontal analysis helps analysts set reasonable expectations for future periods. If COGS has grown at 15% CAGR for five years, assuming it will suddenly drop to 5% requires a specific catalyst (new supplier contract, process improvement, automation investment). Without such a catalyst, trend extrapolation provides a more defensible forecast.

---
## 10. Comprehensive Income

**Comprehensive income** includes net income *plus* items that bypass the income statement and go directly to equity. These are called **Other Comprehensive Income (OCI)** items:

$$
\text{Comprehensive Income} = \text{Net Income} + \text{Other Comprehensive Income}
$$

### Common OCI items

| OCI Item | Example | Why excluded from NI? |
|----------|---------|----------------------|
| Foreign currency translation | Subsidiary's financials translated at new exchange rate | Unrealised, may reverse |
| Unrealised gains/losses on AFS securities | Bond portfolio value changes | Not yet sold |
| Cash flow hedge gains/losses | Interest rate swap fair value changes | Hedging instrument, not core operations |
| Pension plan remeasurements | Actuarial gains/losses from assumption changes | Long-term estimate, volatile |
| Asset revaluation surplus (IFRS only) | Land revalued upward | Unrealised appreciation |

> **Key Concept:** OCI items are *real* economic gains and losses — they affect shareholder wealth. They are excluded from net income because they are unrealised, volatile, or not part of core operations. But ignoring them entirely can lead to a distorted view of total shareholder returns.

> **CFA Exam Tip:** The exam may present a scenario where net income looks strong but comprehensive income is significantly lower (e.g., due to foreign currency losses). You should be able to identify which OCI items are driving the divergence and assess their economic significance.

### "Recycling" — reclassification from OCI to P&L

When an OCI item is *realised* (e.g., the AFS security is sold), it "recycles" — it is reclassified from accumulated OCI on the balance sheet to the income statement. Under IFRS 9, some items (like equity investments at FVOCI) are **never recycled**, creating a permanent bypass of the income statement.

This asymmetry is one of the more nuanced areas of financial reporting and a frequent exam topic.

### Why OCI items bypass the income statement

The decision to route certain items through OCI rather than net income reflects a philosophical debate in accounting:

**Argument for OCI:** These items are unrealised, volatile, and often reverse. Including them in net income would make earnings noisier without adding useful information about operating performance.

**Argument against OCI:** They represent real changes in shareholder wealth. Excluding them from net income allows management to hide significant economic exposures (currency risk, investment losses) in a section of the financial statements that many investors ignore.

> **Key Concept:** Research by Dhaliwal et al. (1999) found that comprehensive income is a better predictor of future cash flows than net income for firms in industries with significant OCI items (banking, insurance, multinational corporations). Ignoring OCI can lead to materially wrong conclusions about a company's true economic performance.

### Worked example — the hidden currency loss

Consider GlobalCo, a US-based multinational:
* Net income: \$500M (looks great — up 10% from prior year)
* Foreign currency translation loss: −\$200M (European subsidiary's financial statements translated at weaker EUR/USD rate)
* Unrealised loss on AFS securities: −\$50M

$$\text{Comprehensive Income} = \$500M - \$200M - \$50M = \$250M$$

While net income shows 10% growth, comprehensive income is only \$250M — **half** the net income figure. An analyst who ignores OCI would miss the fact that the company's total shareholder wealth increased by far less than the earnings headline suggests.

### Where to find OCI

OCI appears in two places:
1. **Statement of Comprehensive Income** — shows the current period's OCI items (the "flow").
2. **Accumulated OCI (AOCI)** in the equity section of the balance sheet — shows the cumulative total of all past OCI items (the "stock").

AOCI is often a large negative number for US multinationals with significant foreign operations and pension obligations. A company with −\$5 billion in AOCI has experienced \$5 billion in unrealised losses that bypassed the income statement — real economic losses that most investors never notice.

---
## 11. Non-Recurring Items and Earnings Quality

Not all income is created equal. Analysts distinguish between **recurring** (sustainable, likely to continue) and **non-recurring** (one-off, unlikely to repeat) items:

### Types of non-recurring items

| Item | Treatment | Effect on Analysis |
|------|-----------|-------------------|
| **Discontinued operations** | Reported separately, net of tax, below continuing operations | Exclude from forward-looking projections |
| **Restructuring charges** | Reported in operating expenses | Exclude if truly one-time; scrutinise if "recurring restructuring" |
| **Asset impairments** | Reported in operating expenses | Signals overvalued assets; exclude from normalised earnings |
| **Litigation settlements** | Operating or non-operating | Exclude unless the company is a serial litigant |
| **Gains/losses on asset sales** | Non-operating | Exclude from operating analysis |

> **Key Concept:** "Normalised earnings" or "adjusted earnings" remove non-recurring items to reveal the *sustainable earning power* of the business. However, companies have an incentive to classify bad news as "non-recurring" while leaving good one-offs in recurring income.

### Big bath accounting

A particularly aggressive tactic: when a company has a bad year anyway, it may recognise *extra* losses (write-downs, restructuring charges) to "clear the decks." This depresses the current year further but makes *future* years look better by comparison. The pattern to watch for:

1. Year of CEO change: massive write-downs → very low earnings
2. Following years: artificially smooth, improving earnings trend

> **Common Mistake:** Do not automatically exclude all restructuring charges. If a company restructures every year, these are effectively recurring operating costs disguised as one-offs.

> **CFA Exam Tip:** The CFA curriculum tests your ability to distinguish between earnings components that should be included vs excluded when forecasting. The question is always: *"Is this likely to recur?"*

### Computing normalised earnings

The process of computing normalised (or "core" or "adjusted") earnings follows a systematic approach:

1. **Start with reported net income**
2. **Add back** non-recurring charges (restructuring, impairments, litigation losses)
3. **Subtract** non-recurring gains (asset sale gains, insurance recoveries, litigation settlements received)
4. **Tax-adjust** each item: Non-recurring item × (1 − marginal tax rate)
5. **Result:** Normalised earnings — the sustainable, recurring profitability of the business

$$\text{Normalised NI} = \text{Reported NI} - \text{After-tax Non-recurring Gains} + \text{After-tax Non-recurring Charges}$$

> **Common Mistake:** Companies often label unfavourable items as "non-recurring" while including favourable one-offs in recurring income. This asymmetric classification inflates normalised earnings. The analyst's job is to apply *consistent* criteria to both positive and negative items — a skill that requires reading the footnotes carefully rather than accepting management's characterisations at face value.

### The "serial restructurer" problem

Some companies take restructuring charges so frequently that they are effectively recurring. If a company has booked restructuring charges in four of the last five years, these are *not* non-recurring — they are a regular cost of doing business and should be included in normalised earnings.

The CFA curriculum specifically warns against accepting management's classification uncritically. The analyst's independent judgment about what is truly one-time versus recurring is a core competency being tested.

### Discontinued operations — a special presentation

When a company sells or shuts down a **major business segment**, the results are reported separately:

$$
\begin{aligned}
&\text{Income from continuing operations} \\
+\;& \text{Income (loss) from discontinued operations, net of tax} \\
=\;& \text{Net income}
\end{aligned}
$$

This segregation is critically important because it separates the **ongoing business** (which will generate future earnings) from the **exiting business** (which will not). Analysts building forward projections should use *continuing operations* as the starting point, not total net income.

> **CFA Exam Tip:** The exam distinguishes carefully between items that are reported "above the line" (within continuing operations, affecting operating or non-operating income) and "below the line" (discontinued operations, reported separately net of tax). Restructuring charges for continuing operations remain above the line; gains/losses from selling a discontinued segment are reported below.

---
## 12. Two-Company Comparison

Let us apply common-size analysis to compare AlphaTech (our technology manufacturer) with **BetaService Inc**, a pure-play software company with a very different cost structure.

This comparison illustrates why common-size analysis is so powerful for cross-sectional analysis: it removes the size difference and reveals fundamental differences in business models.

### Selecting comparable companies

In practice, the most challenging part of cross-sectional analysis is selecting appropriate peers. The ideal comparable company:

1. **Operates in the same industry** — similar products, customers, and competitive dynamics.
2. **Has similar size** — very large companies benefit from economies of scale that distort comparisons.
3. **Uses the same accounting standards** — IFRS vs GAAP differences can create artificial divergences.
4. **Is at a similar lifecycle stage** — comparing a high-growth startup to a mature company is misleading.

When perfect comparables don't exist (and they rarely do), analysts typically use a set of 5-10 companies that are "close enough" and focus on the *range* of ratios rather than any single point estimate.

> **Key Concept:** The purpose of comparable analysis is not to find identical companies — it is to establish a *baseline* against which to judge the target company's performance. Is its gross margin above or below the peer median? Is it gaining or losing ground? The comparison provides context that single-company analysis cannot.

In [ ]:
# ── Two-company common-size comparison ──

# BetaService Inc — 5-year income statement ($M)
# Software company: high gross margins, low capex, SBC-heavy
beta_revenue  = np.array([200.0, 250.0, 310.0, 380.0, 450.0])
beta_cogs     = np.array([ 40.0,  48.0,  56.0,  65.0,  72.0])
beta_sga      = np.array([ 60.0,  72.0,  87.0, 100.0, 113.0])
beta_rd       = np.array([ 50.0,  62.0,  78.0,  95.0, 112.0])
beta_da       = np.array([  8.0,  10.0,  12.0,  15.0,  18.0])
beta_interest = np.array([  2.0,   2.0,   3.0,   4.0,   5.0])
beta_other    = np.array([  1.0,   2.0,   1.0,   3.0,   2.0])
beta_gp       = beta_revenue - beta_cogs
beta_ebit     = beta_gp - beta_sga - beta_rd - beta_da
beta_ebt      = beta_ebit - beta_interest + beta_other
beta_tax      = beta_ebt * 0.25
beta_ni       = beta_ebt - beta_tax

# Common-size comparison for 2024 (latest year)
def cs_compare(label, alpha_val, alpha_rev, beta_val, beta_rev):
    a_pct = alpha_val / alpha_rev * 100
    b_pct = beta_val / beta_rev * 100
    return f"{label:>20}  {a_pct:>7.1f}%  {b_pct:>7.1f}%  {b_pct - a_pct:>+7.1f}pp"

print("Common-Size Comparison — 2024")
print(f"{'':>20}  {'Alpha':>8}  {'Beta':>8}  {'Diff':>9}")
print("─" * 50)
yr = -1  # latest year
comparisons = [
    ('Revenue',      revenue[yr], beta_revenue[yr]),
    ('COGS',         cogs[yr], beta_cogs[yr]),
    ('Gross Profit', gross_profit[yr], beta_gp[yr]),
    ('SG&A',         sga[yr], beta_sga[yr]),
    ('R&D',          rd[yr], beta_rd[yr]),
    ('D&A',          da[yr], beta_da[yr]),
    ('Operating Inc.', operating_inc[yr], beta_ebit[yr]),
    ('Net Income',   net_income[yr], beta_ni[yr]),
]
for label, a_val, b_val in comparisons:
    print(cs_compare(label, a_val, revenue[yr], b_val, beta_revenue[yr]))

print(f"\nAlphaTech Revenue: ${revenue[yr]:.0f}M  |  BetaService Revenue: ${beta_revenue[yr]:.0f}M")

The side-by-side common-size table quantifies the structural differences between the two business models. BetaService (software) has a dramatically higher gross margin but spends far more on R&D and SG&A as a percentage of revenue.

### Why software margins are so high

Software has near-zero **marginal cost** — once the code is written, each additional user costs almost nothing to serve (especially for cloud-delivered SaaS). This creates gross margins of 70-85%, compared to 20-50% for physical goods manufacturers.

However, software companies must spend heavily to:
* **Acquire customers** (SG&A: marketing, sales teams, free trials)
* **Develop products** (R&D: engineers, product managers, infrastructure)
* **Retain customers** (customer success teams, ongoing feature development)

These costs mean that high gross margins do not automatically translate to high net margins — a subtle but critical point that common-size analysis makes visible.

In [ ]:
# ── Visual comparison: side-by-side common-size bars ──

categories = ['COGS', 'SG&A', 'R&D', 'D&A', 'Interest\n+ Tax', 'Net\nIncome']
yr = -1

alpha_pcts = np.array([
    cogs[yr]/revenue[yr], sga[yr]/revenue[yr], rd[yr]/revenue[yr],
    da[yr]/revenue[yr], (interest_exp[yr] + tax_expense[yr])/revenue[yr],
    net_income[yr]/revenue[yr]
]) * 100

beta_pcts = np.array([
    beta_cogs[yr]/beta_revenue[yr], beta_sga[yr]/beta_revenue[yr], beta_rd[yr]/beta_revenue[yr],
    beta_da[yr]/beta_revenue[yr], (beta_interest[yr] + beta_tax[yr])/beta_revenue[yr],
    beta_ni[yr]/beta_revenue[yr]
]) * 100

x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, alpha_pcts, width, label='AlphaTech (Mfg)', color=PRIMARY, edgecolor='white')
bars2 = ax.bar(x + width/2, beta_pcts, width, label='BetaService (SW)', color=SECONDARY, edgecolor='white')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylabel('% of Revenue')
ax.set_title('Common-Size Comparison — AlphaTech (Manufacturing) vs BetaService (Software)')
ax.legend()
plt.tight_layout()
plt.show()

The comparison reveals starkly different business models:

* **BetaService's gross margin (~84%)** dwarfs AlphaTech's (~53%) — software has near-zero marginal cost per unit.
* However, BetaService spends much more on **R&D (25%)** and **SG&A (25%)** as percentages of revenue — software companies must invest heavily in product development and customer acquisition.
* **Net margins** end up closer together than gross margins, illustrating that high gross margins do not automatically translate to high net margins.

### The "margin funnel" concept

Think of the income statement as a funnel: revenue enters at the top, and various costs drain it at each level. AlphaTech has a **wide drain at the top** (high COGS), while BetaService has **wide drains in the middle** (high SG&A and R&D). The funnel metaphor helps analysts quickly characterise a business model:

* **Narrow top drain, wide middle drains** = software, consulting, financial services
* **Wide top drain, narrow middle drains** = commodity manufacturing, retail, distribution
* **Wide drains everywhere** = usually a struggling business or one in heavy investment mode

> **Key Concept:** Common-size analysis strips away the size difference (AlphaTech is \$760M in revenue vs BetaService at \$450M) and reveals the underlying *economic architecture* of each business. This is the foundation of comparable company analysis in valuation.

> **CFA Exam Tip:** The exam may give you common-size income statements for two companies and ask you to identify which company operates in which industry. Practice matching cost structure patterns to industries — it is a recurring question format.

### Implications for valuation

The business model differences revealed by common-size analysis have direct valuation implications:

* **Software companies** (high gross margin, recurring revenue) typically trade at higher EV/Revenue multiples (5-15x) because each incremental dollar of revenue drops a large fraction to the bottom line.
* **Hardware manufacturers** (lower gross margin, cyclical revenue) trade at lower EV/Revenue multiples (1-3x) because revenue growth does not translate as efficiently into profit growth.
* **EV/EBITDA** partially adjusts for this by looking at profitability rather than revenue, but it cannot capture the full difference in margin sustainability and growth potential.

The two-company comparison in this notebook illustrates why analysts must look beyond headline growth rates to the underlying economics — identical revenue growth can produce very different shareholder returns depending on cost structure.

---
## 13. Summary & References

This notebook covered the essential tools for income statement analysis:

* The **multi-step format** provides intermediate profit measures (gross profit, EBIT, EBT, net income), each answering a different analytical question.
* **EBITDA** is widely used but has significant limitations — it ignores capital intensity, working capital, and is not a standardised measure.
* **Basic and diluted EPS** quantify per-share profitability; the treasury stock method handles option dilution by netting exercise proceeds against new shares.
* **Common-size analysis** (vertical) expresses every item as a percentage of revenue, enabling cross-company and cross-period comparison.
* **Horizontal analysis** tracks indexed growth of each line item, revealing divergent growth rates that drive margin trends.
* **Comprehensive income** includes OCI items that bypass the income statement but affect total shareholder wealth.
* **Non-recurring items** must be identified and excluded (or adjusted) when assessing sustainable earning power.

The next notebook explores the **balance sheet** — including inventory methods (FIFO, LIFO, weighted average), depreciation, and working capital analysis.

---

### References

1. CFA Institute. *CFA Program Curriculum Level I*, "Understanding Income Statements."
2. IFRS 15 — *Revenue from Contracts with Customers*.
3. IAS 1 — *Presentation of Financial Statements*.
4. Penman, S. *Financial Statement Analysis and Security Valuation*, 5th ed.
5. Damodaran, A. *Investment Valuation*, 3rd ed., Chapter 3 (Understanding Financial Statements).

### Key formulas to remember

| Formula | Application |
|---------|-------------|
| Gross Margin = (Rev − COGS) / Rev | Production efficiency |
| Operating Margin = EBIT / Rev | Core business profitability |
| Net Margin = NI / Rev | Bottom-line profitability |
| EBITDA = EBIT + D&A | Capital structure-neutral profit |
| Basic EPS = (NI − Pref Div) / WACS | Per-share earnings |
| Diluted EPS = Adj NI / (WACS + Dilutive Shares) | Worst-case per-share |
| Common-size = Line Item / Revenue × 100% | Removes scale for comparison |
| Index = Value_t / Value_base × 100 | Tracks growth from base year |

### CFA exam preparation notes

Income statement analysis is one of the most heavily tested areas within CFA Level 1 Financial Statement Analysis. Key areas of focus:

1. **Multi-step income statement structure** — know every intermediate subtotal and what it reveals
2. **Basic vs diluted EPS** — treasury stock method for options, if-converted method for convertibles
3. **Common-size analysis** — ability to compute and interpret quickly from raw data
4. **EBITDA** — definition, uses, and limitations (especially vs free cash flow)
5. **Revenue recognition** — five-step model, principal vs agent, over time vs point in time
6. **Non-recurring items** — classification, impact on normalised earnings
7. **Comprehensive income** — OCI items and their relationship to net income
8. **IFRS vs GAAP** — R&D capitalisation, expense classification (nature vs function)

The exam typically presents a mini case study with 2-3 years of income statement data and asks analytical questions. Practice reading common-size tables quickly — time pressure makes speed essential.

> **Final note:** Mastery of the concepts in this notebook is essential for the CFA Level 1 exam, as well as for practical financial analysis work. Practice the worked examples by hand and verify your understanding by reproducing the code from scratch.
